# MealMate: Leftover Food Recipe Generator

Run the cells in order from top to bottom. Each writefile cell creates one project file. The last cell starts the server and prints a public URL.


In [ ]:
# CELL 1: Install packages and create folders
!pip install -q fastapi uvicorn httpx google-auth pyngrok
!mkdir -p /content/mealmate/backend /content/mealmate/frontend
print("Ready. Run the next cells in order.")


In [ ]:
%%writefile /content/mealmate/backend/main.py
import os
import re
import json
import time
import sqlite3
import secrets
import hashlib
import urllib.parse
from typing import Optional, List

import httpx
from fastapi import FastAPI, HTTPException, Header
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
from google.oauth2 import id_token as google_id_token
from google.auth.transport import requests as google_requests

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
FRONTEND_DIR = os.path.join(BASE_DIR, "..", "frontend")
DB_PATH = os.path.join(BASE_DIR, "mealmate.db")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
GOOGLE_CLIENT_ID = os.environ.get("GOOGLE_CLIENT_ID", "")
AFFILIATE_IDS = {
    "instacart": os.environ.get("AFFILIATE_INSTACART", ""),
    "jumia": os.environ.get("AFFILIATE_JUMIA", ""),
    "amazon": os.environ.get("AFFILIATE_AMAZON", ""),
}

app = FastAPI(title="MealMate")

# ---------------- Database (SQLite; swap for Supabase/Firebase later) ----------------

SCHEMA = """
CREATE TABLE IF NOT EXISTS users (
    id TEXT PRIMARY KEY,
    email TEXT UNIQUE NOT NULL,
    pw_hash TEXT DEFAULT '',
    name TEXT DEFAULT '',
    google_sub TEXT DEFAULT '',
    is_premium INTEGER DEFAULT 0,
    diet TEXT DEFAULT '',
    created REAL
);
CREATE TABLE IF NOT EXISTS sessions (
    token TEXT PRIMARY KEY,
    user_id TEXT,
    created REAL
);
CREATE TABLE IF NOT EXISTS favorites (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT,
    recipe_json TEXT,
    created REAL
);
CREATE TABLE IF NOT EXISTS shopping (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT,
    item TEXT,
    done INTEGER DEFAULT 0,
    created REAL
);
"""


def get_db():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db()
    conn.executescript(SCHEMA)
    conn.close()


def run(sql, params=()):
    conn = get_db()
    try:
        cur = conn.execute(sql, params)
        conn.commit()
        return cur.lastrowid
    finally:
        conn.close()


def fetch(sql, params=(), one=False):
    conn = get_db()
    try:
        rows = [dict(r) for r in conn.execute(sql, params).fetchall()]
    finally:
        conn.close()
    if one:
        return rows[0] if rows else None
    return rows


init_db()

# ---------------- Auth helpers ----------------


def hash_password(password, salt=None):
    salt = salt or secrets.token_hex(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt.encode(), 200000).hex()
    return f"{salt}${digest}"


def verify_password(password, stored):
    if not stored or "$" not in stored:
        return False
    salt = stored.split("$", 1)[0]
    return secrets.compare_digest(hash_password(password, salt), stored)


def create_session(user_id):
    token = secrets.token_urlsafe(32)
    run("INSERT INTO sessions(token, user_id, created) VALUES(?,?,?)", (token, user_id, time.time()))
    return token


def public_user(user_id):
    u = fetch("SELECT id, email, name, is_premium, diet FROM users WHERE id=?", (user_id,), one=True)
    if u:
        u["is_premium"] = bool(u["is_premium"])
    return u


def get_user_id(authorization):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Please sign in to use this feature.")
    row = fetch("SELECT user_id FROM sessions WHERE token=?", (authorization[7:],), one=True)
    if not row:
        raise HTTPException(401, "Your session has expired. Please sign in again.")
    return row["user_id"]


def optional_user(authorization):
    try:
        return public_user(get_user_id(authorization))
    except HTTPException:
        return None


# ---------------- Schemas ----------------


class SignUp(BaseModel):
    email: str
    password: str
    name: str = ""


class SignIn(BaseModel):
    email: str
    password: str


class GoogleAuth(BaseModel):
    credential: str


class Prefs(BaseModel):
    diet: str = ""


class RecipeRequest(BaseModel):
    ingredients: List[str]
    servings: int = 2
    max_time: int = 45
    diet: str = ""
    count: int = 3


class FavoriteIn(BaseModel):
    recipe: dict


class ShopItems(BaseModel):
    items: List[str]


# ---------------- Recipe generation ----------------

RECIPE_PROMPT = """
You are a creative chef who helps people reduce food waste.
Create <<COUNT>> recipes that mainly use these ingredients: <<INGREDIENTS>>.
Assume basic pantry staples (salt, pepper, cooking oil, water, sugar) are available.
Put any other needed item under missing_ingredients and suggest substitutes that use the listed ingredients when possible.
Constraints: servings = <<SERVINGS>>, total time at most <<MAX_TIME>> minutes, dietary preference = <<DIET>>.
Return ONLY valid JSON in this exact shape:
{
  "recipes": [
    {
      "title": "string",
      "emoji": "one food emoji",
      "image_keyword": "two word food keyword for a photo search",
      "description": "one short sentence",
      "cook_time_minutes": 0,
      "servings": 0,
      "difficulty": "easy or medium or hard",
      "ingredients_used": ["string"],
      "missing_ingredients": [{"name": "string", "substitutes": ["string"]}],
      "steps": ["string"],
      "nutrition": {"calories": 0, "protein_g": 0, "carbs_g": 0, "fat_g": 0, "fiber_g": 0},
      "health_tips": ["string"],
      "portion_tip": "string"
    }
  ]
}
"""


def clean_ingredients(items):
    out = []
    for item in items:
        for part in re.split(r"[,\n]", item):
            name = re.sub(r"\s+", " ", part).strip()[:60]
            if name and name.lower() not in [o.lower() for o in out]:
                out.append(name)
    return out[:40]


async def generate_recipes(ingredients, servings, max_time, diet, count):
    if not OPENAI_API_KEY:
        raise HTTPException(500, "Server is missing OPENAI_API_KEY.")

    prompt = (
        RECIPE_PROMPT.replace("<<COUNT>>", str(count))
        .replace("<<INGREDIENTS>>", ", ".join(ingredients))
        .replace("<<SERVINGS>>", str(servings))
        .replace("<<MAX_TIME>>", str(max_time))
        .replace("<<DIET>>", diet)
    )
    payload = {
        "model": OPENAI_MODEL,
        "temperature": 0.9,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": "You are a creative chef. Reply with JSON only."},
            {"role": "user", "content": prompt},
        ],
    }
    async with httpx.AsyncClient(timeout=60) as client:
        res = await client.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
            json=payload,
        )
    if res.status_code != 200:
        raise HTTPException(502, "The recipe service is busy. Try again.")
    try:
        data = json.loads(res.json()["choices"][0]["message"]["content"])
        return data.get("recipes", [])[:count]
    except (KeyError, ValueError, AttributeError):
        raise HTTPException(502, "Could not read the recipe response. Try again.")


# ---------------- Routes ----------------


@app.get("/api/config")
def config():
    return {"google_client_id": GOOGLE_CLIENT_ID}


@app.post("/api/auth/signup")
def signup(body: SignUp):
    email = body.email.strip().lower()
    if "@" not in email or len(body.password) < 8:
        raise HTTPException(400, "Enter a valid email and a password of at least 8 characters.")
    uid = secrets.token_hex(8)
    try:
        run(
            "INSERT INTO users(id, email, pw_hash, name, created) VALUES(?,?,?,?,?)",
            (uid, email, hash_password(body.password), body.name.strip()[:60], time.time()),
        )
    except sqlite3.IntegrityError:
        raise HTTPException(409, "An account with this email already exists.")
    return {"token": create_session(uid), "user": public_user(uid)}


@app.post("/api/auth/signin")
def signin(body: SignIn):
    email = body.email.strip().lower()
    row = fetch("SELECT id, pw_hash FROM users WHERE email=?", (email,), one=True)
    if not row or not verify_password(body.password, row["pw_hash"]):
        raise HTTPException(401, "Wrong email or password.")
    return {"token": create_session(row["id"]), "user": public_user(row["id"])}


@app.post("/api/auth/google")
def google_auth(body: GoogleAuth):
    if not GOOGLE_CLIENT_ID:
        raise HTTPException(500, "Google sign-in is not configured.")
    try:
        info = google_id_token.verify_oauth2_token(
            body.credential, google_requests.Request(), GOOGLE_CLIENT_ID
        )
    except Exception:
        raise HTTPException(401, "Google sign-in failed. Try again.")

    email = info["email"].lower()
    row = fetch("SELECT id FROM users WHERE email=?", (email,), one=True)
    if row:
        uid = row["id"]
    else:
        uid = secrets.token_hex(8)
        run(
            "INSERT INTO users(id, email, pw_hash, name, google_sub, created) VALUES(?,?,?,?,?,?)",
            (uid, email, "", info.get("name", "")[:60], info["sub"], time.time()),
        )
    return {"token": create_session(uid), "user": public_user(uid)}


@app.get("/api/me")
def me(authorization: Optional[str] = Header(None)):
    return public_user(get_user_id(authorization))


@app.post("/api/me/preferences")
def set_preferences(body: Prefs, authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    run("UPDATE users SET diet=? WHERE id=?", (body.diet[:30], uid))
    return public_user(uid)


@app.post("/api/premium/activate")
def activate_premium(authorization: Optional[str] = Header(None)):
    # DEMO ONLY: unlocks Premium without payment.
    # In production, replace this with a Stripe or Paystack webhook that sets is_premium=1.
    uid = get_user_id(authorization)
    run("UPDATE users SET is_premium=1 WHERE id=?", (uid,))
    return public_user(uid)


@app.post("/api/recipes")
async def create_recipes(body: RecipeRequest, authorization: Optional[str] = Header(None)):
    ingredients = clean_ingredients(body.ingredients)
    if not ingredients:
        raise HTTPException(400, "Add at least one ingredient.")

    user = optional_user(authorization)
    is_premium = bool(user and user["is_premium"])
    diet = body.diet or (user["diet"] if user else "") or "none"
    count = min(max(body.count, 1), 5)

    recipes = await generate_recipes(ingredients, body.servings, body.max_time, diet, count)
    for r in recipes:
        r["nutrition_locked"] = not is_premium
        if not is_premium:
            r["nutrition"] = None
            r["health_tips"] = []
    return {"recipes": recipes, "show_ads": not is_premium}


@app.get("/api/favorites")
def list_favorites(authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    rows = fetch(
        "SELECT id, recipe_json, created FROM favorites WHERE user_id=? ORDER BY created DESC",
        (uid,),
    )
    return {
        "favorites": [
            {"id": r["id"], "recipe": json.loads(r["recipe_json"]), "created": r["created"]}
            for r in rows
        ]
    }


@app.post("/api/favorites")
def add_favorite(body: FavoriteIn, authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    fid = run(
        "INSERT INTO favorites(user_id, recipe_json, created) VALUES(?,?,?)",
        (uid, json.dumps(body.recipe), time.time()),
    )
    return {"id": fid}


@app.delete("/api/favorites/{fid}")
def delete_favorite(fid: int, authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    run("DELETE FROM favorites WHERE id=? AND user_id=?", (fid, uid))
    return {"ok": True}


@app.get("/api/shopping")
def list_shopping(authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    return {"items": fetch("SELECT id, item, done FROM shopping WHERE user_id=? ORDER BY done, id", (uid,))}


@app.post("/api/shopping")
def add_shopping(body: ShopItems, authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    existing = {
        r["item"].lower()
        for r in fetch("SELECT item FROM shopping WHERE user_id=? AND done=0", (uid,))
    }
    added = 0
    for raw in body.items:
        name = raw.strip()[:60]
        if name and name.lower() not in existing:
            run(
                "INSERT INTO shopping(user_id, item, done, created) VALUES(?,?,0,?)",
                (uid, name, time.time()),
            )
            existing.add(name.lower())
            added += 1
    return {"added": added}


@app.post("/api/shopping/{sid}/toggle")
def toggle_shopping(sid: int, authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    run("UPDATE shopping SET done = 1 - done WHERE id=? AND user_id=?", (sid, uid))
    return {"ok": True}


@app.delete("/api/shopping/{sid}")
def delete_shopping(sid: int, authorization: Optional[str] = Header(None)):
    uid = get_user_id(authorization)
    run("DELETE FROM shopping WHERE id=? AND user_id=?", (sid, uid))
    return {"ok": True}


@app.get("/api/affiliate")
def affiliate_links(q: str = ""):
    term = urllib.parse.quote_plus(q.strip() or "groceries")

    def with_tag(url, key, param):
        return f"{url}&{param}={AFFILIATE_IDS[key]}" if AFFILIATE_IDS[key] else url

    links = [
        {"store": "Instacart", "url": with_tag(f"https://www.instacart.com/store/s?k={term}", "instacart", "aff")},
        {"store": "Jumia Food", "url": with_tag(f"https://www.jumia.com.ng/catalog/?q={term}", "jumia", "aff")},
        {"store": "Amazon Fresh", "url": with_tag(f"https://www.amazon.com/s?k={term}&i=amazonfresh", "amazon", "tag")},
    ]
    return {"links": links}


# Serve the frontend last so API routes take priority
app.mount("/", StaticFiles(directory=FRONTEND_DIR, html=True), name="web")


In [ ]:
%%writefile /content/mealmate/frontend/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover" />
<meta name="theme-color" content="#2F7D5B" />
<meta name="apple-mobile-web-app-capable" content="yes" />
<title>MealMate</title>
<link rel="manifest" href="/manifest.json" />
<link rel="icon" href="/icon.svg" />
<link rel="apple-touch-icon" href="/icon.svg" />
<link href="https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;800&display=swap" rel="stylesheet" />
<script src="https://accounts.google.com/gsi/client" async defer></script>
<style>
:root { --green:#2F7D5B; --mint:#E8F5EE; --sun:#F6B93B; --ink:#1F2A24; --muted:#6B7A72; --card:#fff; --line:#DCE7E1; }
* { box-sizing: border-box; }
html, body { margin: 0; }
body { font-family: Nunito, system-ui, sans-serif; background: var(--mint); color: var(--ink); min-height: 100vh; padding-bottom: 84px; }
.app { max-width: 720px; margin: 0 auto; padding: 0 16px; }
.top { display: flex; justify-content: space-between; align-items: center; padding: 16px 0; }
.brand { font-weight: 800; font-size: 22px; color: var(--green); }
main section { padding-top: 8px; }
h1 { font-size: 28px; margin: 12px 0 4px; }
h2 { font-size: 22px; margin: 8px 0 12px; }
.sub { color: var(--muted); margin: 4px 0 12px; }
.input-row { display: flex; gap: 8px; }
input, select { font: inherit; padding: 12px 14px; border: 1px solid var(--line); border-radius: 12px; background: #fff; width: 100%; color: var(--ink); }
.input-row input { flex: 1; }
button { font: inherit; cursor: pointer; border-radius: 12px; border: 0; padding: 10px 14px; }
.primary { background: var(--green); color: #fff; font-weight: 700; padding: 12px 16px; }
.primary:disabled { opacity: .6; }
.big { width: 100%; margin-top: 16px; padding: 16px; font-size: 17px; }
.small { padding: 8px 12px; font-size: 14px; }
.ghost { background: #fff; color: var(--green); border: 1px solid var(--line); }
.icon-btn { background: #fff; border: 1px solid var(--line); font-size: 20px; padding: 0 14px; }
.link { background: none; color: var(--green); padding: 8px 0; width: 100%; }
.chips { display: flex; flex-wrap: wrap; gap: 8px; margin: 12px 0; }
.chip { background: #fff; border: 1px solid var(--line); border-radius: 999px; padding: 6px 6px 6px 12px; display: flex; align-items: center; gap: 6px; }
.chip button { background: var(--mint); padding: 0 8px; border-radius: 999px; color: var(--ink); }
.options { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-top: 8px; }
.options label, .stack { display: flex; flex-direction: column; gap: 6px; font-size: 14px; color: var(--muted); font-weight: 600; }
.error { color: #C0392B; font-weight: 600; }
.card { background: var(--card); border-radius: 18px; overflow: hidden; margin: 14px 0; box-shadow: 0 4px 14px rgba(0,0,0,.06); }
.card-img { position: relative; height: 180px; background: var(--mint); display: flex; align-items: center; justify-content: center; }
.card-img .emoji { font-size: 64px; }
.card-img img { position: absolute; inset: 0; width: 100%; height: 100%; object-fit: cover; }
.card-body { padding: 14px 16px 16px; }
.card-body h3 { margin: 0 0 4px; font-size: 19px; }
.meta { display: flex; gap: 10px; flex-wrap: wrap; font-size: 13px; color: var(--muted); margin: 8px 0; }
.actions { display: flex; gap: 8px; flex-wrap: wrap; }
.details { border-top: 1px solid var(--line); margin-top: 12px; padding-top: 8px; font-size: 15px; line-height: 1.5; }
.details h4 { margin: 14px 0 6px; color: var(--green); }
.nutri { display: grid; grid-template-columns: repeat(5, 1fr); gap: 6px; text-align: center; }
.nutri b { display: block; font-size: 17px; }
.nutri span { font-size: 12px; color: var(--muted); }
.locked { background: var(--mint); padding: 12px; border-radius: 12px; }
.stores { display: flex; gap: 8px; flex-wrap: wrap; }
.stores a { background: var(--sun); color: var(--ink); padding: 8px 12px; border-radius: 10px; text-decoration: none; font-weight: 700; }
.ad { background: #FFF8E1; border: 1px dashed var(--sun); border-radius: 12px; padding: 12px; text-align: center; font-size: 14px; }
.fav { background: #fff; border-radius: 14px; padding: 12px 14px; margin: 10px 0; display: flex; justify-content: space-between; gap: 12px; align-items: center; }
.fav p { margin: 4px 0 0; }
.check { display: flex; gap: 10px; align-items: center; }
.check input { width: auto; }
.done { text-decoration: line-through; color: var(--muted); }
.pill { background: var(--sun); border-radius: 999px; padding: 4px 10px; font-weight: 700; font-size: 13px; }
.bottom { position: fixed; bottom: 0; left: 0; right: 0; background: #fff; display: flex; justify-content: space-around; padding: 10px 6px calc(10px + env(safe-area-inset-bottom, 0px)); border-top: 1px solid var(--line); }
.bottom button { background: none; color: var(--muted); font-weight: 700; padding: 6px 10px; }
.bottom button.active { color: var(--green); }
.toast { position: fixed; left: 50%; transform: translateX(-50%); bottom: 90px; background: var(--ink); color: #fff; padding: 10px 16px; border-radius: 999px; }
.auth { text-align: center; }
.or { color: var(--muted); margin: 14px 0; }
.premium { background: #fff; border-radius: 16px; padding: 16px; margin: 16px 0; }
@media (max-width: 480px) { .options { grid-template-columns: 1fr 1fr; } .nutri { grid-template-columns: repeat(3, 1fr); } }
</style>
</head>
<body>
<div id="root"></div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/react/18.3.1/umd/react.production.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/react-dom/18.3.1/umd/react-dom.production.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/babel-standalone/7.24.7/babel.min.js"></script>
<script type="text/babel">
const { useState, useEffect, useRef } = React;

const DIETS = ["none", "vegan", "vegetarian", "low-carb", "gluten-free", "dairy-free", "keto"];

async function api(path, { method = "GET", body, token } = {}) {
  const headers = { "Content-Type": "application/json" };
  if (token) headers.Authorization = "Bearer " + token;
  const res = await fetch(path, { method, headers, body: body ? JSON.stringify(body) : undefined });
  const data = await res.json().catch(() => ({}));
  if (!res.ok) throw new Error(typeof data.detail === "string" ? data.detail : "Something went wrong. Try again.");
  return data;
}

function splitIngredients(text) {
  return text.split(/[,\n]|\band\b/i).map(s => s.trim()).filter(Boolean);
}

function mergeUnique(base, extra) {
  const out = [...base];
  extra.forEach(x => {
    if (!out.some(o => o.toLowerCase() === x.toLowerCase())) out.push(x);
  });
  return out;
}

function listen(onText, setListening) {
  const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  if (!SR) { alert("Voice input is not supported in this browser."); return; }
  const rec = new SR();
  rec.lang = "en-US";
  rec.interimResults = false;
  rec.maxAlternatives = 1;
  setListening(true);
  rec.onresult = e => onText(e.results[0][0].transcript);
  rec.onerror = () => setListening(false);
  rec.onend = () => setListening(false);
  rec.start();
}

function Home(p) {
  const [listening, setListening] = useState(false);

  const addDraft = () => {
    if (!p.draft.trim()) return;
    p.setIngredients(prev => mergeUnique(prev, splitIngredients(p.draft)));
    p.setDraft("");
  };
  const onKey = e => {
    if (e.key === "Enter" || e.key === ",") { e.preventDefault(); addDraft(); }
  };

  return (
    <section>
      <h1>What is in your kitchen?</h1>
      <p className="sub">Turn leftovers into real meals.</p>
      <div className="input-row">
        <input
          value={p.draft}
          onChange={e => p.setDraft(e.target.value)}
          onKeyDown={onKey}
          placeholder="Enter your ingredients..."
        />
        <button className="icon-btn" title="Voice input"
          onClick={() => listen(t => p.setIngredients(prev => mergeUnique(prev, splitIngredients(t))), setListening)}>
          {listening ? "..." : "🎤"}
        </button>
      </div>
      <div className="chips">
        {p.ingredients.map(i => (
          <span key={i} className="chip">
            {i}
            <button onClick={() => p.setIngredients(p.ingredients.filter(x => x !== i))}>×</button>
          </span>
        ))}
      </div>
      <div className="options">
        <label>Diet
          <select value={p.diet} onChange={e => p.setDiet(e.target.value)}>
            {DIETS.map(d => <option key={d} value={d === "none" ? "" : d}>{d}</option>)}
          </select>
        </label>
        <label>Servings
          <input type="number" min="1" max="10" value={p.servings} onChange={e => p.setServings(e.target.value)} />
        </label>
        <label>Max minutes
          <select value={p.maxTime} onChange={e => p.setMaxTime(e.target.value)}>
            {[15, 30, 45, 60, 90].map(m => <option key={m} value={m}>{m}</option>)}
          </select>
        </label>
      </div>
      {p.error && <p className="error">{p.error}</p>}
      <button className="primary big" disabled={p.loading} onClick={() => p.onGenerate(p.draft)}>
        {p.loading ? "Cooking up ideas..." : "Generate Recipes"}
      </button>
    </section>
  );
}

function RecipeCard({ recipe, token, onNeedAuth, onNotice, onUpgrade }) {
  const [open, setOpen] = useState(false);
  const [imgOk, setImgOk] = useState(true);
  const [links, setLinks] = useState(null);
  const missing = recipe.missing_ingredients || [];
  const n = recipe.nutrition;

  const save = async () => {
    if (!token) return onNeedAuth();
    try {
      await api("/api/favorites", { method: "POST", token, body: { recipe } });
      onNotice("Saved to favorites");
    } catch (e) { onNotice(e.message); }
  };

  const share = async () => {
    const text = recipe.title + ": " + recipe.description;
    if (navigator.share) {
      try { await navigator.share({ title: recipe.title, text }); } catch (e) {}
    } else {
      try { await navigator.clipboard.writeText(text); onNotice("Copied to clipboard"); }
      catch (e) { onNotice("Sharing is not available here"); }
    }
  };

  const addMissing = async () => {
    if (!token) return onNeedAuth();
    if (!missing.length) return onNotice("You have everything you need");
    try {
      await api("/api/shopping", { method: "POST", token, body: { items: missing.map(m => m.name) } });
      onNotice("Added to your list");
    } catch (e) { onNotice(e.message); }
  };

  const loadLinks = async () => {
    const q = missing.length ? missing.map(m => m.name).join(" ") : recipe.title;
    try {
      const d = await api("/api/affiliate?q=" + encodeURIComponent(q));
      setLinks(d.links);
    } catch (e) { onNotice(e.message); }
  };

  const keyword = (recipe.image_keyword || "food").replace(/\s+/g, ",");

  return (
    <article className="card">
      <div className="card-img">
        <span className="emoji">{recipe.emoji || "🍽️"}</span>
        {imgOk && (
          <img
            src={"https://loremflickr.com/640/360/" + encodeURIComponent(keyword)}
            alt={recipe.title}
            onError={() => setImgOk(false)}
          />
        )}
      </div>
      <div className="card-body">
        <h3>{recipe.title}</h3>
        <p className="sub">{recipe.description}</p>
        <div className="meta">
          <span>⏱ {recipe.cook_time_minutes} min</span>
          <span>🍽 {recipe.servings} servings</span>
          <span>{recipe.difficulty}</span>
        </div>
        <div className="actions">
          <button className="ghost small" onClick={() => setOpen(!open)}>{open ? "Hide" : "View recipe"}</button>
          <button className="ghost small" onClick={save}>Save</button>
          <button className="ghost small" onClick={share}>Share</button>
        </div>

        {open && (
          <div className="details">
            <h4>Ingredients you have</h4>
            <p>{(recipe.ingredients_used || []).join(", ")}</p>

            {missing.length > 0 && (
              <>
                <h4>Missing, with substitutes</h4>
                <ul>
                  {missing.map((m, i) => (
                    <li key={i}>
                      <strong>{m.name}</strong>
                      {m.substitutes && m.substitutes.length ? " (try " + m.substitutes.join(" or ") + ")" : ""}
                    </li>
                  ))}
                </ul>
                <button className="primary small" onClick={addMissing}>Add missing to list</button>
              </>
            )}

            <h4>Steps</h4>
            <ol>{(recipe.steps || []).map((s, i) => <li key={i}>{s}</li>)}</ol>

            <h4>Nutrition</h4>
            {n ? (
              <>
                <div className="nutri">
                  {[["Calories", n.calories], ["Protein g", n.protein_g], ["Carbs g", n.carbs_g], ["Fat g", n.fat_g], ["Fiber g", n.fiber_g]]
                    .map(([k, v]) => <div key={k}><b>{v}</b><span>{k}</span></div>)}
                </div>
                {recipe.health_tips && recipe.health_tips.length > 0 && (
                  <ul>{recipe.health_tips.map((t, i) => <li key={i}>{t}</li>)}</ul>
                )}
              </>
            ) : recipe.nutrition_locked ? (
              <div className="locked">
                <p>Nutrition facts and health tips are a Premium feature.</p>
                <button className="primary small" onClick={onUpgrade}>Unlock Premium</button>
              </div>
            ) : null}

            {recipe.portion_tip && <p><b>Portion tip:</b> {recipe.portion_tip}</p>}

            <h4>Buy missing ingredients</h4>
            {!links ? (
              <button className="ghost small" onClick={loadLinks}>Show store links</button>
            ) : (
              <div className="stores">
                {links.map(l => (
                  <a key={l.store} href={l.url} target="_blank" rel="sponsored noopener noreferrer">{l.store}</a>
                ))}
              </div>
            )}
          </div>
        )}
      </div>
    </article>
  );
}

function AuthScreen({ onAuthed }) {
  const [mode, setMode] = useState("signin");
  const [name, setName] = useState("");
  const [email, setEmail] = useState("");
  const [password, setPassword] = useState("");
  const [err, setErr] = useState("");
  const gbtn = useRef(null);

  useEffect(() => {
    let timer = null;
    const init = cid => {
      window.google.accounts.id.initialize({
        client_id: cid,
        callback: r => api("/api/auth/google", { method: "POST", body: { credential: r.credential } })
          .then(d => onAuthed(d.token))
          .catch(e => setErr(e.message)),
      });
      window.google.accounts.id.renderButton(gbtn.current, { theme: "outline", size: "large", width: 280, text: "continue_with" });
    };
    api("/api/config").then(cfg => {
      if (!cfg.google_client_id) return;
      timer = setInterval(() => {
        if (window.google && gbtn.current) { clearInterval(timer); init(cfg.google_client_id); }
      }, 300);
    }).catch(() => {});
    return () => clearInterval(timer);
  }, []);

  const submit = async e => {
    e.preventDefault();
    setErr("");
    try {
      const d = await api(mode === "signup" ? "/api/auth/signup" : "/api/auth/signin", {
        method: "POST",
        body: { name, email, password },
      });
      onAuthed(d.token);
    } catch (ex) { setErr(ex.message); }
  };

  return (
    <section className="auth">
      <h2>{mode === "signup" ? "Create account" : "Sign in"}</h2>
      <p className="sub">Save recipes and shopping lists on any device.</p>
      <div ref={gbtn}></div>
      <p className="or">or with email</p>
      <form onSubmit={submit} className="stack">
        {mode === "signup" && (
          <input placeholder="Name" value={name} onChange={e => setName(e.target.value)} />
        )}
        <input type="email" required placeholder="Email" value={email} onChange={e => setEmail(e.target.value)} />
        <input type="password" required minLength={8} placeholder="Password (8+ characters)" value={password} onChange={e => setPassword(e.target.value)} />
        <button className="primary">{mode === "signup" ? "Create account" : "Sign in"}</button>
      </form>
      {err && <p className="error">{err}</p>}
      <button className="link" onClick={() => setMode(mode === "signup" ? "signin" : "signup")}>
        {mode === "signup" ? "Have an account? Sign in" : "New here? Create an account"}
      </button>
    </section>
  );
}

function Account({ user, token, onSignOut, onUpdate, notify }) {
  const [diet, setDiet] = useState(user.diet || "");

  const save = async () => {
    try {
      const u = await api("/api/me/preferences", { method: "POST", token, body: { diet } });
      onUpdate(u);
      notify("Diet profile saved");
    } catch (e) { notify(e.message); }
  };

  const upgrade = async () => {
    try {
      const u = await api("/api/premium/activate", { method: "POST", token });
      onUpdate(u);
      notify("Premium unlocked");
    } catch (e) { notify(e.message); }
  };

  return (
    <section>
      <h2>{user.name || user.email}</h2>
      <p className="sub">{user.email}</p>
      <div className="stack">
        <label className="stack">Dietary profile
          <select value={diet} onChange={e => setDiet(e.target.value)}>
            {DIETS.map(d => <option key={d} value={d === "none" ? "" : d}>{d}</option>)}
          </select>
        </label>
        <button className="primary" onClick={save}>Save profile</button>
      </div>
      {!user.is_premium ? (
        <div className="premium">
          <h3>Go Premium</h3>
          <p className="sub">Nutrition facts, health tips, and no ads.</p>
          <button className="primary" onClick={upgrade}>Unlock Premium</button>
        </div>
      ) : (
        <p><span className="pill">Premium active</span></p>
      )}
      <button className="ghost" onClick={onSignOut}>Sign out</button>
    </section>
  );
}

function App() {
  const [token, setToken] = useState(localStorage.getItem("mm_token") || "");
  const [user, setUser] = useState(null);
  const [view, setView] = useState("home");
  const [ingredients, setIngredients] = useState([]);
  const [draft, setDraft] = useState("");
  const [diet, setDiet] = useState("");
  const [servings, setServings] = useState(2);
  const [maxTime, setMaxTime] = useState(45);
  const [recipes, setRecipes] = useState([]);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState("");
  const [toast, setToast] = useState("");
  const [favorites, setFavorites] = useState([]);
  const [shopping, setShopping] = useState([]);
  const [newItem, setNewItem] = useState("");

  const showAds = !(user && user.is_premium);

  const notify = msg => {
    setToast(msg);
    setTimeout(() => setToast(""), 2500);
  };

  useEffect(() => {
    if (!token) { setUser(null); return; }
    api("/api/me", { token }).then(setUser).catch(() => {
      localStorage.removeItem("mm_token");
      setToken("");
    });
  }, [token]);

  const loadFavorites = async () => {
    try { const d = await api("/api/favorites", { token }); setFavorites(d.favorites); }
    catch (e) { notify(e.message); }
  };

  const loadShopping = async () => {
    try { const d = await api("/api/shopping", { token }); setShopping(d.items); }
    catch (e) { notify(e.message); }
  };

  useEffect(() => {
    if (!token) return;
    if (view === "favorites") loadFavorites();
    if (view === "shopping") loadShopping();
  }, [view, token]);

  const generate = async draftText => {
    const list = mergeUnique(ingredients, splitIngredients(draftText || ""));
    if (!list.length) { setError("Add at least one ingredient."); return; }
    setIngredients(list);
    setDraft("");
    setError("");
    setLoading(true);
    setView("results");
    try {
      const data = await api("/api/recipes", {
        method: "POST",
        token,
        body: { ingredients: list, servings: Number(servings), max_time: Number(maxTime), diet, count: 3 },
      });
      setRecipes(data.recipes || []);
    } catch (e) {
      setError(e.message);
      setView("home");
    } finally {
      setLoading(false);
    }
  };

  const upgrade = async () => {
    if (!token) return setView("account");
    try {
      const u = await api("/api/premium/activate", { method: "POST", token });
      setUser(u);
      notify("Premium unlocked");
    } catch (e) { notify(e.message); }
  };

  const handleAuthed = t => {
    localStorage.setItem("mm_token", t);
    setToken(t);
    setView("home");
  };

  const signOut = () => {
    localStorage.removeItem("mm_token");
    setToken("");
    setUser(null);
    setView("home");
  };

  const needAuth = () => {
    setView("account");
    notify("Sign in to save and sync");
  };

  const SignInPrompt = () => (
    <div className="premium">
      <p className="sub">Sign in to use this feature.</p>
      <button className="primary" onClick={() => setView("account")}>Sign in</button>
    </div>
  );

  return (
    <div className="app">
      <header className="top">
        <div className="brand">🥗 MealMate</div>
        {user && user.is_premium && <span className="pill">Premium</span>}
      </header>

      <main>
        {view === "home" && (
          <Home
            ingredients={ingredients} setIngredients={setIngredients}
            draft={draft} setDraft={setDraft}
            diet={diet} setDiet={setDiet}
            servings={servings} setServings={setServings}
            maxTime={maxTime} setMaxTime={setMaxTime}
            onGenerate={generate} loading={loading} error={error}
          />
        )}

        {view === "results" && (
          <section>
            {showAds && <div className="ad">Sponsored. Go Premium for an ad-free experience.</div>}
            <h2>Your recipes</h2>
            {loading && <p className="sub">Finding recipes that use your ingredients...</p>}
            {recipes.map((r, i) => (
              <RecipeCard key={i} recipe={r} token={token}
                onNeedAuth={needAuth} onNotice={notify} onUpgrade={upgrade} />
            ))}
            {!loading && recipes.length === 0 && <p className="sub">No recipes yet. Add ingredients and try again.</p>}
            {!loading && <button className="ghost big" onClick={() => setView("home")}>Start over</button>}
          </section>
        )}

        {view === "favorites" && (
          <section>
            <h2>Favorites</h2>
            {!token ? <SignInPrompt /> : favorites.length === 0 ? (
              <p className="sub">No saved recipes yet.</p>
            ) : favorites.map(f => (
              <div className="fav" key={f.id}>
                <div>
                  <strong>{f.recipe.emoji} {f.recipe.title}</strong>
                  <p className="sub">{f.recipe.description}</p>
                </div>
                <button className="ghost small" onClick={async () => {
                  try { await api("/api/favorites/" + f.id, { method: "DELETE", token }); loadFavorites(); }
                  catch (e) { notify(e.message); }
                }}>Remove</button>
              </div>
            ))}
          </section>
        )}

        {view === "shopping" && (
          <section>
            <h2>Shopping list</h2>
            {!token ? <SignInPrompt /> : (
              <>
                <form className="input-row" onSubmit={async e => {
                  e.preventDefault();
                  const items = splitIngredients(newItem);
                  if (!items.length) return;
                  try {
                    await api("/api/shopping", { method: "POST", token, body: { items } });
                    setNewItem("");
                    loadShopping();
                  } catch (err) { notify(err.message); }
                }}>
                  <input value={newItem} onChange={e => setNewItem(e.target.value)} placeholder="Add an item..." />
                  <button className="primary">Add</button>
                </form>
                {shopping.map(s => (
                  <div className="fav" key={s.id}>
                    <label className="check">
                      <input type="checkbox" checked={!!s.done} onChange={async () => {
                        try { await api("/api/shopping/" + s.id + "/toggle", { method: "POST", token }); loadShopping(); }
                        catch (e) { notify(e.message); }
                      }} />
                      <span className={s.done ? "done" : ""}>{s.item}</span>
                    </label>
                    <button className="ghost small" onClick={async () => {
                      try { await api("/api/shopping/" + s.id, { method: "DELETE", token }); loadShopping(); }
                      catch (e) { notify(e.message); }
                    }}>Remove</button>
                  </div>
                ))}
                {shopping.length === 0 && <p className="sub">Your list is empty.</p>}
              </>
            )}
          </section>
        )}

        {view === "account" && (
          user ? (
            <Account user={user} token={token} onSignOut={signOut} onUpdate={setUser} notify={notify} />
          ) : (
            <AuthScreen onAuthed={handleAuthed} />
          )
        )}
      </main>

      <nav className="bottom">
        {[["home", "Home"], ["favorites", "Saved"], ["shopping", "List"], ["account", "Account"]].map(([k, l]) => (
          <button key={k} className={view === k ? "active" : ""} onClick={() => setView(k)}>{l}</button>
        ))}
      </nav>

      {toast && <div className="toast">{toast}</div>}
    </div>
  );
}

ReactDOM.createRoot(document.getElementById("root")).render(<App />);
</script>
<script>
  if ("serviceWorker" in navigator) navigator.serviceWorker.register("/sw.js");
</script>
</body>
</html>


In [ ]:
%%writefile /content/mealmate/frontend/manifest.json
{
  "name": "MealMate",
  "short_name": "MealMate",
  "start_url": "/",
  "display": "standalone",
  "background_color": "#E8F5EE",
  "theme_color": "#2F7D5B",
  "icons": [
    { "src": "/icon.svg", "sizes": "any", "type": "image/svg+xml", "purpose": "any maskable" }
  ]
}


In [ ]:
%%writefile /content/mealmate/frontend/sw.js
const CACHE = "mealmate-v1";
const SHELL = ["/", "/index.html", "/manifest.json", "/icon.svg"];

self.addEventListener("install", e => {
  e.waitUntil(caches.open(CACHE).then(c => c.addAll(SHELL)));
  self.skipWaiting();
});

self.addEventListener("activate", e => {
  e.waitUntil(self.clients.claim());
});

self.addEventListener("fetch", e => {
  const url = new URL(e.request.url);
  if (e.request.method !== "GET" || url.origin !== self.location.origin || url.pathname.startsWith("/api/")) return;
  e.respondWith(
    fetch(e.request)
      .then(res => {
        const copy = res.clone();
        caches.open(CACHE).then(c => c.put(e.request, copy));
        return res;
      })
      .catch(() => caches.match(e.request))
  );
});


In [ ]:
%%writefile /content/mealmate/frontend/icon.svg
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512">
  <rect width="512" height="512" rx="110" fill="#2F7D5B"/>
  <circle cx="256" cy="270" r="150" fill="#E8F5EE"/>
  <circle cx="256" cy="270" r="100" fill="#F6B93B"/>
  <path d="M256 170c-60 0-90 40-90 90 60-5 90-40 90-90z" fill="#2F7D5B"/>
</svg>


In [ ]:
%%writefile /content/mealmate/README.md
# MealMate: Leftover Food Recipe Generator

Turn leftover ingredients into recipes. Reduce food waste.

## Stack
- Backend: FastAPI (Python), SQLite (swap for Supabase or Firebase later)
- AI: OpenAI Chat Completions (JSON mode)
- Frontend: React 18 PWA, installable on mobile and desktop
- Auth: Email/password and Google Sign-In

## Run in Google Colab
1. Run the setup cell.
2. Create the backend and frontend files by running each writefile cell.
3. Run the launch cell and enter your keys when prompted.
4. Open the printed public URL on your phone or laptop.

## Environment variables
- OPENAI_API_KEY: required for recipe generation
- OPENAI_MODEL: optional, defaults to gpt-4o-mini
- GOOGLE_CLIENT_ID: optional, enables Google Sign-In
- AFFILIATE_INSTACART, AFFILIATE_JUMIA, AFFILIATE_AMAZON: optional affiliate IDs

## Google Sign-In setup
Add your public URL as an Authorized JavaScript origin in Google Cloud Console (OAuth client ID, Web application).

## Production notes
- Replace /api/premium/activate with a Stripe or Paystack webhook.
- Move data from SQLite to Supabase or Firestore for persistence across Colab restarts.
- Host the backend on Render, Railway, or Cloud Run for a permanent URL.


In [ ]:
# CELL: Launch MealMate
import os
import sys
import time
import threading
from getpass import getpass

import uvicorn
from pyngrok import ngrok

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
os.environ["GOOGLE_CLIENT_ID"] = input("Google OAuth client ID (press Enter to skip): ").strip()
# Optional affiliate IDs:
# os.environ["AFFILIATE_INSTACART"] = "your-id"
# os.environ["AFFILIATE_JUMIA"] = "your-id"
# os.environ["AFFILIATE_AMAZON"] = "your-tag"

ngrok.set_auth_token(getpass("ngrok auth token: "))

sys.path.insert(0, "/content/mealmate/backend")
from main import app

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

tunnel = ngrok.connect(8000)
print("MealMate is live at:", tunnel.public_url)
